# FYP Data Analysis (v2)

**ESG Disclosure & Investment Efficiency** — reorganized notebook.

```
=
FYP Data Analysis (v2) — ESG Disclosure & Investment Efficiency
Author: Jiayu Hu

REORGANIZED LOGIC (vs. the previous script)
-------------------------------------------------------------------------------------
 - Lags are built ONCE on the full firm panel (Section 10), then sliced into every
   subsample, so the t-1 value is correct at subsample boundaries.
 - The lag follows the mediator TRANS:
      * No TRANS in the regression  -> contemporaneous (H1 Tables 5-7; 2SLS/GMM Table 10)
      * TRANS in the regression     -> one-year lag (ESGR_L1/ESGID_L1 + TRANS_L1)
 - H2 mediation (Table 8) is reported in three steps:
      Step 1  Baron & Kenny, contemporaneous           -> TRANS NOT significant
      Step 2  Baron & Kenny, ESGR & TRANS both lagged   -> significant (esp. IE1)
      Step 3  Bootstrap of the indirect effect a*b      -> confirms Step 2
 - ESGID robustness (Table 9) and all heterogeneity tables (11-15) use the SAME
   lagged mediation spec as Step 2.
 - Every regression printout shows ONLY the focal coefficients (controls + FE are
   estimated but hidden).

ENVIRONMENT
  pip install pandas "numpy<2" statsmodels linearmodels pydynpd openpyxl scipy xlrd
  (pydynpd / system GMM needs numpy < 2; use 1.26.x.)

Runs section-by-section as `# %%` cells in VS Code / Jupyter, or all at once.
=
```

### [0] Environment & dependencies

In [1]:
# %% [0] Environment & dependencies ---------------------------------------------------
# Run once to install dependencies (pydynpd/system GMM needs numpy<2)
# %pip install pandas "numpy<2" statsmodels linearmodels pydynpd openpyxl scipy xlrd

import os
import warnings
import numpy as np
import pandas as pd

# pydynpd still calls np.in1d (removed in numpy 2.0); harmless shim under numpy<2.
if not hasattr(np, "in1d"):
    np.in1d = np.isin  # noqa

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.iv import IV2SLS
from scipy.stats import norm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

WORKDIR = os.environ.get("FYP_DIR", ".")
os.chdir(WORKDIR)

### [0.1] Helper functions

In [2]:
# %% [0.1] Helper functions -----------------------------------------------------------
def winsorize(s: pd.Series, lower_pct: float = 0.01, upper_pct: float = 0.99) -> pd.Series:
    """1%/99% winsorization (type-7 quantiles, matches pandas default)."""
    lo, hi = s.quantile(lower_pct), s.quantile(upper_pct)
    return s.clip(lower=lo, upper=hi)


def pad_symbol(s: pd.Series) -> pd.Series:
    """Zero-pad the stock code to 6 chars (R's sprintf('%06s', Symbol))."""
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(6)


# Control variables (consistent with the report)
CONTROLS = ["Size", "PB", "Lev", "ROA", "Zsorcing", "LA", "Board", "IDR",
            "Age", "Female", "FB", "OB", "TOP", "SECOND", "DUAL", "SOE"]


def fe_ols(df, y, x_vars, controls=None, fe=("Industry", "Year"),
           cluster="Symbol", add_const=True):
    """
    OLS with Year FE + Industry FE (reghdfe/areg equivalent), SE clustered by firm.
    Returns a statsmodels results object.
    """
    controls = controls or []
    sub = df.dropna(subset=[y] + list(x_vars) + list(controls) + list(fe)).copy()
    for f in fe:
        sub[f] = sub[f].astype("category")
    rhs = list(x_vars) + list(controls) + list(fe)
    formula = f"{y} ~ " + " + ".join(rhs)
    if not add_const:
        formula += " - 1"
    if cluster and cluster in sub.columns:
        res = smf.ols(formula, data=sub).fit(
            cov_type="cluster", cov_kwds={"groups": sub[cluster]})
    else:
        res = smf.ols(formula, data=sub).fit(cov_type="HC1")
    return res


def report(res, focal, label=""):
    """Print ONLY the focal coefficients (controls + FE are hidden)."""
    focal = list(focal)
    rows = []
    for v in res.params.index:
        base = v.split("[")[0]
        if base in focal:
            rows.append((v, res.params[v], res.tvalues[v], res.pvalues[v]))
    out = pd.DataFrame(rows, columns=["var", "coef", "t", "p"]).set_index("var")
    out["sig"] = pd.cut(out["p"], [-1, .01, .05, .1, 1], labels=["***", "**", "*", ""])
    if label:
        print(f"\n=== {label} | N={int(res.nobs)} | adj_R2={res.rsquared_adj:.4f} ===")
    print(out.round(4))
    return out.round(4)


def add_firm_lags(df, cols, k=1, by="Symbol", year="Year"):
    """
    Create {col}_L{k} within firm, ONLY across genuinely consecutive years.
    Build lags on the FULL panel before any subsetting.
    """
    d = df.sort_values([by, year]).copy()
    prev = d.groupby(by)[year].shift(k)
    consecutive = (d[year] - prev == k)
    for c in cols:
        d[f"{c}_L{k}"] = d.groupby(by)[c].shift(k).where(consecutive)
    return d


def boot_mediation(df, y, x, m, controls=CONTROLS, fe=("Industry", "Year"),
                   cluster="Symbol", n_boot=1000, seed=42):
    """
    Mediation X -> M -> Y with FE + clustered SE.
      path a : M ~ X (+controls+FE)
      path b : Y ~ X + M (+controls+FE)      (b = coef on M; direct = coef on X)
      indirect = a*b
    Reports: point estimate, Sobel z/p, and a firm-cluster bootstrap percentile CI.
    Controls are kept contemporaneous; X and M are whatever columns you pass
    (e.g. ESGR_L1 and TRANS_L1 for the lagged mediation).
    """
    need = [y, x, m] + list(controls) + list(fe) + [cluster]
    sub = df.dropna(subset=need).copy()

    # ---- point estimates + SE (clustered) ----
    ra = fe_ols(sub, m, [x], controls, fe, cluster)
    rb = fe_ols(sub, y, [x, m], controls, fe, cluster)
    a, sa = ra.params[x], ra.bse[x]
    b, sb = rb.params[m], rb.bse[m]
    direct = rb.params[x]
    indirect = a * b

    # ---- Sobel test ----
    sobel_se = np.sqrt(b**2 * sa**2 + a**2 * sb**2)
    sobel_z = indirect / sobel_se if sobel_se > 0 else np.nan
    sobel_p = 2 * (1 - norm.cdf(abs(sobel_z))) if np.isfinite(sobel_z) else np.nan

    # ---- fast design matrices (build once) for the cluster bootstrap ----
    fe_dum = pd.get_dummies(sub[list(fe)].astype(str), drop_first=True).astype(float)
    Cm = sub[controls].astype(float)
    const = pd.Series(1.0, index=sub.index, name="const")
    Xa = pd.concat([const, sub[[x]].astype(float), Cm, fe_dum], axis=1)
    Xb = pd.concat([const, sub[[x]].astype(float), sub[[m]].astype(float), Cm, fe_dum], axis=1)
    Xa_v, Xb_v = Xa.values, Xb.values
    yv, mv = sub[y].values.astype(float), sub[m].values.astype(float)
    ia_x = Xa.columns.get_loc(x)     # position of X in path a
    ib_m = Xb.columns.get_loc(m)     # position of M in path b

    codes = sub[cluster].values
    uniq = np.unique(codes)
    firm_rows = {f: np.where(codes == f)[0] for f in uniq}
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        rows = np.concatenate([firm_rows[f] for f in pick])
        ba, *_ = np.linalg.lstsq(Xa_v[rows], mv[rows], rcond=None)
        bb, *_ = np.linalg.lstsq(Xb_v[rows], yv[rows], rcond=None)
        boots[i] = ba[ia_x] * bb[ib_m]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p_boot = 2 * min((boots <= 0).mean(), (boots >= 0).mean())

    print(f"\n--- Bootstrap mediation: {x} -> {m} -> {y} | N={int(rb.nobs)} ---")
    print(f"  a (X->M)          = {a:.4f}  (SE={sa:.4f})")
    print(f"  b (M->Y|X)        = {b:.4f}  (SE={sb:.4f})")
    print(f"  direct (X->Y|M)   = {direct:.4f}")
    print(f"  indirect a*b      = {indirect:.6f}")
    print(f"  Sobel z           = {sobel_z:.3f}  (p={sobel_p:.4f})")
    print(f"  Bootstrap 95% CI  = [{lo:.6f}, {hi:.6f}]  (reps={n_boot})")
    print(f"  Bootstrap p       = {p_boot:.4f}  "
          f"{'-> significant (CI excludes 0)' if lo*hi > 0 else '-> not significant (CI includes 0)'}")
    return dict(a=a, b=b, direct=direct, indirect=indirect,
                sobel_z=sobel_z, sobel_p=sobel_p, ci=(lo, hi), p_boot=p_boot)


# =====================================================================================
# DATA PREPARATION (unchanged pipeline)
# =====================================================================================

### [1] Read data

In [3]:
# %% [1] Read data --------------------------------------------------------------------
Rechardson = pd.read_excel("Rechardson.xlsx")
Biddle     = pd.read_excel("Biddle.xlsx")
Chen       = pd.read_excel("Chen.xlsx")
ESG_IV     = pd.read_excel("华证ESG及工具变量.xlsx")    # Sino-Securities ESG & instruments
CV         = pd.read_excel("Control Variables.xlsx")
ZS         = pd.read_excel("Zsorcing.xlsx")
IA         = pd.read_excel("ASY.xlsx")
ESG34      = pd.read_excel("ESG分歧34.xls")             # ESG rating divergence (6 agencies)

### [2] Cleaning & type conversion

In [4]:
# %% [2] Cleaning & type conversion ---------------------------------------------------
Rechardson = Rechardson.drop_duplicates(subset=["Symbol", "Year"], keep="first").copy()


def year_to_int(s):
    dt = pd.to_datetime(s, errors="coerce")
    return dt.dt.year.fillna(pd.to_numeric(s, errors="coerce")).astype("Int64").astype(int)


Rechardson["Year"] = year_to_int(Rechardson["Year"])
Rechardson["Symbol"] = pad_symbol(Rechardson["Symbol"])

for d in (Biddle, Chen, CV, ESG34):
    d["Symbol"] = pad_symbol(d["Symbol"])
    d["Year"] = pd.to_numeric(d["Year"], errors="coerce").astype("Int64").astype(int)

ZS["Year"] = year_to_int(ZS["Year"])
ZS["Symbol"] = pad_symbol(ZS["Symbol"])

for d in (IA, ESG_IV):
    if "Symbol" in d.columns:
        d["Symbol"] = pad_symbol(d["Symbol"])
    d["Year"] = pd.to_numeric(d["Year"], errors="coerce").astype("Int64").astype(int)

### [3] Year filter 2013-2023

In [5]:
# %% [3] Year filter 2013-2023 --------------------------------------------------------
def yrfilter(d):
    return d[(d["Year"] >= 2013) & (d["Year"] <= 2023)].copy()


Rechardson1, Biddle1, Chen1 = map(yrfilter, (Rechardson, Biddle, Chen))
CV1, ZS1, IA1 = map(yrfilter, (CV, ZS, IA))
ESG_IV1, ESG341 = map(yrfilter, (ESG_IV, ESG34))

### [4] Select & rename variables

In [6]:
# %% [4] Select & rename variables ----------------------------------------------------
Rechardson2 = (Rechardson1[["Symbol", "Year", "IndustryName",
                            "InefficInvestDegree", "InefficInvestSign"]]
               .rename(columns={"InefficInvestDegree": "IE1",
                                "InefficInvestSign": "Over_or_Under_Investment"}))

Biddle2 = Biddle1[["Symbol", "Year", "Inveffi"]].rename(columns={"Inveffi": "IE2"})
Chen2   = Chen1[["Symbol", "Year", "Inveffi"]].rename(columns={"Inveffi": "IE3"})

CV2 = (CV1[["Symbol", "Year", "Size", "PB", "Lev", "ROA1", "ListAge", "Boardsize",
            "IndDirectorRatio", "AverageAge", "MaleRatio", "MngmFinancialBack",
            "MngmOverseaBack", "Shrcr1", "Shrz", "Dual", "ContrshrNature", "TobinQ"]]
       .rename(columns={"ContrshrNature": "SOE", "ROA1": "ROA", "ListAge": "LA",
                        "Boardsize": "Board", "IndDirectorRatio": "IDR",
                        "AverageAge": "Age", "MngmFinancialBack": "FB",
                        "MngmOverseaBack": "OB", "Shrcr1": "TOP", "Dual": "DUAL"}))

ZS2 = ZS1[["Symbol", "Year", "Zsorcing"]].copy()
IA2 = IA1[["Symbol", "Year", "LR", "ILL", "GAM"]].copy()
ESG_IV2 = (ESG_IV1[["Symbol", "Year", "ESG评级赋值",
                    "mean1", "mean2", "mean3", "mean4", "mean5",
                    "mean6", "mean7", "mean8", "mean9", "mean10", "IndustryCode"]]
           .rename(columns={"IndustryCode": "Industry"}))

ESG342 = ESG341[["Symbol", "Year", "ESGdif6", "ESGmin1", "ESGmax1", "ESGrange6"]].copy()

### [5] Impute ZS with firm mean

In [7]:
# %% [5] Impute ZS with firm mean -----------------------------------------------------
ZS2["Zsorcing"] = ZS2.groupby("Symbol")["Zsorcing"].transform(lambda x: x.fillna(x.mean()))
ZS2 = ZS2.groupby("Symbol").filter(lambda g: g["Zsorcing"].notna().any())

### [6] Merge

In [8]:
# %% [6] Merge ------------------------------------------------------------------------
def lj(left, right):
    m = left.merge(right, on=["Symbol", "Year"], how="left")
    return m.drop_duplicates(subset=["Symbol", "Year"], keep="first")


merged = Rechardson2
for r in (CV2, Biddle2, Chen2, ZS2, IA2, ESG_IV2, ESG342):
    merged = lj(merged, r)

### [7] Drop missing, negate IE, build Female / SECOND

In [9]:
# %% [7] Drop missing, negate IE, build Female / SECOND -------------------------------
Statement = merged.dropna().copy()
Statement[["IE1", "IE2", "IE3"]] = -Statement[["IE1", "IE2", "IE3"]]
Statement["Female"] = 100 - Statement["MaleRatio"]
Statement["SECOND"] = (1 / Statement["Shrz"]) * Statement["TOP"]

### [8] Winsorize & ESGR alias

In [10]:
# %% [8] Winsorize & ESGR alias -------------------------------------------------------
wins_cols = (["IE1", "IE2", "IE3", "TobinQ", "ESG评级赋值",
              "ESGdif6", "ESGmin1", "ESGmax1", "ESGrange6"]
             + ["Size", "PB", "Lev", "ROA", "Zsorcing", "LA", "Board", "IDR",
                "Age", "Female", "MaleRatio", "TOP", "SECOND"]
             + [f"mean{i}" for i in range(1, 11)])
for c in wins_cols:
    if c in Statement.columns:
        Statement[c] = winsorize(Statement[c])

Statement["ESGR"] = Statement["ESG评级赋值"]

### [9] Information-asymmetry PCA -> TRANS (= -PC1)

In [11]:
# %% [9] Information-asymmetry PCA -> TRANS (= -PC1) -----------------------------------
# PCA on LR/ILL/GAM; PC1 captures the common information-asymmetry component
# (all three load positively). TRANS = -PC1, so higher TRANS = higher transparency.
def run_asym_pca(df, cols=("LR", "ILL", "GAM")):
    X = df[list(cols)].astype(float)
    Z = (X - X.mean()) / X.std(ddof=0)
    C = np.corrcoef(Z.values, rowvar=False)
    vals, vecs = np.linalg.eigh(C)
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:, order]
    loadings = pd.DataFrame(vecs, index=list(cols), columns=["PC1", "PC2", "PC3"])
    # sign fix: PC1 loads positively (align with Table 1a), PC2 negative on LR
    if loadings.loc["ILL", "PC1"] < 0:
        loadings["PC1"] *= -1
    if loadings.loc["LR", "PC2"] > 0:
        loadings["PC2"] *= -1
    scores = Z.values @ loadings.values
    expl = vals / vals.sum()
    var_tbl = pd.DataFrame({"Eigenvalue": vals,
                            "Prop.%": expl * 100,
                            "Cum.%": np.cumsum(expl) * 100},
                           index=["PC1", "PC2", "PC3"]).round(2)
    print("Information-asymmetry PCA (Table 1a/1b):")
    print(var_tbl)
    print("\nComponent loadings:")
    print(loadings.round(4))
    return scores, loadings


_scores, _loadings = run_asym_pca(Statement)
Statement["ASY1"] = _scores[:, 0]               # PC1 = information asymmetry
Statement["TRANS"] = -Statement["ASY1"]         # transparency = -PC1

Information-asymmetry PCA (Table 1a/1b):
     Eigenvalue  Prop.%   Cum.%
PC1         2.2   73.35   73.35
PC2         0.6   19.98   93.32
PC3         0.2    6.68  100.00

Component loadings:
        PC1     PC2     PC3
LR   0.5761 -0.5836 -0.5723
ILL  0.6297 -0.1296  0.7660
GAM  0.5212  0.8016 -0.2928


### [10] *** Build firm-level lags ONCE on the full panel ***

In [12]:
# %% [10] *** Build firm-level lags ONCE on the full panel *** ------------------------
# This is the central change. Build ESGR_L1 / TRANS_L1 on the complete firm time series
# (consecutive years only), THEN slice into subsamples. Never lag after subsetting.
Statement = add_firm_lags(Statement, ["ESGR", "TRANS"], k=1)
print(f"\nFull sample N = {len(Statement)} | "
      f"lagged (L1) sample N = {Statement.dropna(subset=['ESGR_L1', 'TRANS_L1']).shape[0]}")


# =====================================================================================
# DESCRIPTIVES
# =====================================================================================


Full sample N = 19103 | lagged (L1) sample N = 15064


### [11] Descriptive statistics -- Table 2

In [13]:
# %% [11] Descriptive statistics -- Table 2 -------------------------------------------
def describe_table(df, cols):
    d = df[cols].agg(["mean", "std", "min",
                      lambda s: s.quantile(.25), "median",
                      lambda s: s.quantile(.75), "max"]).T
    d.columns = ["Mean", "SD", "Min", "1stQu", "Median", "3rdQu", "Max"]
    return d.round(2)


print("\n========== Table 2: descriptive statistics ==========")
print(f"N = {len(Statement)}")
print(describe_table(Statement, ["IE1", "IE2", "IE3", "ESGR", "TRANS"] + CONTROLS))


========== Table 2: descriptive statistics ==========
N = 19103
           Mean     SD    Min  1stQu  Median  3rdQu    Max
IE1       -0.04   0.05  -0.27  -0.05   -0.02  -0.01  -0.00
IE2       -0.04   0.04  -0.22  -0.05   -0.03  -0.01  -0.00
IE3       -0.04   0.04  -0.22  -0.05   -0.03  -0.01  -0.00
ESGR       4.20   1.01   1.00   4.00    4.00   5.00   6.00
TRANS     -0.00   1.48 -18.54  -0.80    0.05   0.87   7.54
Size      22.46   1.33  20.13  21.50   22.25  23.20  26.67
PB         3.32   2.75   0.47   1.63    2.53   3.99  17.10
Lev        0.44   0.20   0.07   0.27    0.43   0.58   0.89
ROA        0.03   0.06  -0.23   0.01    0.03   0.07   0.21
Zsorcing   4.75   5.37  -0.02   1.80    3.05   5.49  34.38
LA         2.30   0.69   1.10   1.79    2.30   2.89   3.40
Board      8.37   1.63   5.00   7.00    9.00   9.00  14.00
IDR       37.93   5.40  33.33  33.33   36.36  42.86  57.14
Age       49.78   3.20  42.00  47.64   49.85  52.00  57.34
Female    20.75  11.36   0.00  12.50   20.00  28.5

### [12] Correlation matrix -- Table 3

In [14]:
# %% [12] Correlation matrix -- Table 3 -----------------------------------------------
print("\n========== Table 3: Pearson correlation ==========")
print(Statement[["IE1", "IE2", "IE3", "ESGR"] + CONTROLS].corr().round(2))


========== Table 3: Pearson correlation ==========
           IE1   IE2   IE3  ESGR  Size    PB   Lev   ROA  Zsorcing    LA  Board   IDR   Age  Female    FB    OB   TOP  SECOND  DUAL   SOE
IE1       1.00  0.46  0.45  0.10  0.13 -0.23  0.07 -0.08     -0.12  0.13   0.05 -0.02  0.14   -0.02 -0.04 -0.04  0.02   -0.00 -0.04  0.03
IE2       0.46  1.00  0.98  0.05  0.05 -0.10  0.02 -0.07     -0.02  0.09   0.03 -0.02  0.06   -0.00 -0.01  0.00  0.00    0.00 -0.05  0.00
IE3       0.45  0.98  1.00  0.04  0.04 -0.09  0.02 -0.08     -0.02  0.10   0.03 -0.02  0.06   -0.00 -0.01  0.00  0.00    0.00 -0.05  0.01
ESGR      0.10  0.05  0.04  1.00  0.25 -0.14 -0.07  0.20      0.02 -0.03   0.03  0.07  0.15   -0.02 -0.05 -0.00  0.09    0.06 -0.02  0.03
Size      0.13  0.05  0.04  0.25  1.00 -0.38  0.50  0.05     -0.37  0.41   0.28  0.00  0.37   -0.22  0.11  0.07  0.22    0.04 -0.19  0.21
PB       -0.23 -0.10 -0.09 -0.14 -0.38  1.00 -0.04  0.06      0.44 -0.18  -0.11  0.03 -0.20    0.08 -0.01  0.04 -0.10   

### [13] VIF -- Table 4

In [15]:
# %% [13] VIF -- Table 4 --------------------------------------------------------------
def vif_table(df, xvars):
    X = sm.add_constant(df[xvars].astype(float).dropna())
    out = pd.DataFrame(
        {"VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]},
        index=X.columns)
    return out.drop("const").round(2)


print("\n========== Table 4: VIF ==========")
print(vif_table(Statement, ["ESGR"] + CONTROLS))


# =====================================================================================
# H1: ESG DISCLOSURE -> INVESTMENT EFFICIENCY (contemporaneous; no mediator)
# =====================================================================================


========== Table 4: VIF ==========
           VIF
ESGR      1.17
Size      2.45
PB        1.75
Lev       2.72
ROA       1.31
Zsorcing  2.37
LA        1.50
Board     1.66
IDR       1.47
Age       1.32
Female    1.12
FB        1.13
OB        1.12
TOP       1.17
SECOND    1.12
DUAL      1.13
SOE       1.10


### [14] H1 -- Tables 5-7

In [16]:
# %% [14] H1 -- Tables 5-7 ------------------------------------------------------------
print("\n########## H1: ESGR -> IE (Tables 5-7) ##########")
for ie, tbl in [("IE1", "Table 5"), ("IE2", "Table 6"), ("IE3", "Table 7")]:
    report(fe_ols(Statement, ie, ["ESGR"], controls=None),
           focal=["ESGR"], label=f"{tbl} {ie} | Model 1 (no controls)")
    report(fe_ols(Statement, ie, ["ESGR"], controls=CONTROLS),
           focal=["ESGR"], label=f"{tbl} {ie} | Model 2 (with controls)")


# =====================================================================================
# H2: MEDIATION OF INFORMATION TRANSPARENCY (Table 8)
#   Step 1  contemporaneous Baron & Kenny  -> TRANS not significant
#   Step 2  one-year lag (ESGR_L1 + TRANS_L1) -> significant
#   Step 3  bootstrap of the indirect effect
# =====================================================================================


########## H1: ESGR -> IE (Tables 5-7) ##########

=== Table 5 IE1 | Model 1 (no controls) | N=19103 | adj_R2=0.0974 ===
        coef       t    p  sig
var                           
ESGR  0.0035  9.1447  0.0  ***

=== Table 5 IE1 | Model 2 (with controls) | N=19103 | adj_R2=0.1412 ===
        coef       t    p  sig
var                           
ESGR  0.0029  7.5218  0.0  ***

=== Table 6 IE2 | Model 1 (no controls) | N=19103 | adj_R2=0.0596 ===
        coef       t       p sig
var                             
ESGR  0.0008  2.4623  0.0138  **

=== Table 6 IE2 | Model 2 (with controls) | N=19103 | adj_R2=0.0789 ===
        coef       t       p sig
var                             
ESGR  0.0008  2.4381  0.0148  **

=== Table 7 IE3 | Model 1 (no controls) | N=19103 | adj_R2=0.0576 ===
        coef       t       p sig
var                             
ESGR  0.0005  1.5786  0.1144    

=== Table 7 IE3 | Model 2 (with controls) | N=19103 | adj_R2=0.0777 ===
        coef      t       p sig
va

### [15.1] Step 1 -- Baron & Kenny, CONTEMPORANEOUS

In [17]:
# %% [15.1] Step 1 -- Baron & Kenny, CONTEMPORANEOUS ---------------------------------
print("\n########## H2 Step 1: Baron & Kenny (contemporaneous) ##########")
print("Expectation: path a (ESGR->TRANS) significant, but path b (TRANS->IE) NOT.")
report(fe_ols(Statement, "TRANS", ["ESGR"], CONTROLS),
       focal=["ESGR"], label="Table 8 Model 1: TRANS ~ ESGR  (path a)")
for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
    report(fe_ols(Statement, ie, ["ESGR", "TRANS"], CONTROLS),
           focal=["ESGR", "TRANS"], label=f"Table 8 {mlbl}: {ie} ~ ESGR + TRANS")


########## H2 Step 1: Baron & Kenny (contemporaneous) ##########
Expectation: path a (ESGR->TRANS) significant, but path b (TRANS->IE) NOT.

=== Table 8 Model 1: TRANS ~ ESGR  (path a) | N=19103 | adj_R2=0.6270 ===
        coef       t       p  sig
var                              
ESGR  0.0275  2.9424  0.0033  ***

=== Table 8 Model 2: IE1 ~ ESGR + TRANS | N=19103 | adj_R2=0.1412 ===
         coef       t       p  sig
var                               
ESGR   0.0029  7.5089  0.0000  ***
TRANS  0.0004  1.0819  0.2793     

=== Table 8 Model 3: IE2 ~ ESGR + TRANS | N=19103 | adj_R2=0.0790 ===
         coef       t       p sig
var                              
ESGR   0.0008  2.4045  0.0162  **
TRANS  0.0004  1.1772  0.2391    

=== Table 8 Model 4: IE3 ~ ESGR + TRANS | N=19103 | adj_R2=0.0777 ===
         coef       t       p sig
var                              
ESGR   0.0007  1.9554  0.0505   *
TRANS  0.0003  0.8977  0.3694    


### [15.2] Step 2 -- Baron & Kenny, ONE-YEAR LAG (both ESGR and TRANS)

In [18]:
# %% [15.2] Step 2 -- Baron & Kenny, ONE-YEAR LAG (both ESGR and TRANS) ---------------
# Chain: ESGR_{t-1} -> TRANS_{t-1} -> IE_t. Path a and path b use the SAME mediator
# object (TRANS_L1), so the indirect effect a*b is internally consistent.
print("\n########## H2 Step 2: Baron & Kenny (one-year lag: ESGR_L1 + TRANS_L1) ##########")
report(fe_ols(Statement, "TRANS_L1", ["ESGR_L1"], CONTROLS),
       focal=["ESGR_L1"], label="Table 8 (lag) Model 1: TRANS_L1 ~ ESGR_L1  (path a)")
for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
    report(fe_ols(Statement, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
           focal=["ESGR_L1", "TRANS_L1"],
           label=f"Table 8 (lag) {mlbl}: {ie} ~ ESGR_L1 + TRANS_L1")


########## H2 Step 2: Baron & Kenny (one-year lag: ESGR_L1 + TRANS_L1) ##########

=== Table 8 (lag) Model 1: TRANS_L1 ~ ESGR_L1  (path a) | N=15064 | adj_R2=0.5870 ===
           coef       t       p  sig
var                                 
ESGR_L1  0.0272  2.6312  0.0085  ***

=== Table 8 (lag) Model 2: IE1 ~ ESGR_L1 + TRANS_L1 | N=15064 | adj_R2=0.1561 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0026  6.2827  0.0000  ***
TRANS_L1  0.0011  2.6607  0.0078  ***

=== Table 8 (lag) Model 3: IE2 ~ ESGR_L1 + TRANS_L1 | N=15064 | adj_R2=0.0842 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0016  4.6626  0.0000  ***
TRANS_L1  0.0004  1.1388  0.2548     

=== Table 8 (lag) Model 4: IE3 ~ ESGR_L1 + TRANS_L1 | N=15064 | adj_R2=0.0825 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0014  3.9744  0.0001  ***
TRANS_L1  0.0004  1.2548  0.2096     


### [15.3] Step 3 -- Bootstrap of the indirect effect (lagged spec)

In [19]:
# %% [16] Table 9 -- ESGID with iterative (MICE/EM-style) imputation, lagged ----------
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from statsmodels.multivariate.pca import PCA as smPCA

print("\n########## Table 9: ESGID (iterative imputation + PCA, lagged mediation) ##########")
AGENCY_COLS  = ["华证", "Wind", "富时罗素", "盟浪", "商道融绿", "msci"]  
VAR_THRESHOLD = 0.70   

if all(c in ESG341.columns for c in AGENCY_COLS):
    _ag = ESG341[["Symbol", "Year"] + AGENCY_COLS].drop_duplicates(["Symbol", "Year"])
    St9 = Statement.merge(_ag, on=["Symbol", "Year"], how="left")
    St9 = St9[St9[AGENCY_COLS].notna().any(axis=1)].copy()
    print(f"firm-years kept (>=1 rating) = {len(St9)} | "
          f"missing rating cells imputed = {int(St9[AGENCY_COLS].isna().sum().sum())}")

    imp = IterativeImputer(max_iter=50, random_state=0)
    filled = pd.DataFrame(imp.fit_transform(St9[AGENCY_COLS]),
                          columns=AGENCY_COLS, index=St9.index)

    pc = smPCA(filled, ncomp=len(AGENCY_COLS), standardize=True, method="eig")
    prop = np.asarray(pc.eigenvals).ravel(); prop = prop / prop.sum()
    cum = np.cumsum(prop)

    k = int(np.argmax(cum >= VAR_THRESHOLD)) + 1 if (cum >= VAR_THRESHOLD).any() else len(prop)
    w = prop[:k] / prop[:k].sum()                  
    esgid = pc.factors.iloc[:, :k].values @ w      

    if np.corrcoef(esgid, St9["ESGR"])[0, 1] < 0:
        esgid = -esgid
    St9["ESGID"] = winsorize(pd.Series(esgid, index=St9.index))

    print(pd.DataFrame({"Prop.%": prop * 100, "Cum.%": cum * 100},
                       index=[f"PC{i+1}" for i in range(len(AGENCY_COLS))]).round(2))
    print(f"Components retained (cum >= {VAR_THRESHOLD:.0%}): k = {k} | weights = {np.round(w, 3)}")
    print(f"corr(ESGID, ESGR) = {np.corrcoef(St9['ESGID'], St9['ESGR'])[0, 1]:.3f}")

    St9 = add_firm_lags(St9, ["ESGID"], k=1)
    print(f"ESGID lagged sample N = {St9.dropna(subset=['ESGID_L1', 'TRANS_L1']).shape[0]}")

    report(fe_ols(St9, "TRANS_L1", ["ESGID_L1"], CONTROLS),
           focal=["ESGID_L1"], label="Table 9 (lag) Model 1: TRANS_L1 ~ ESGID_L1")
    for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
        report(fe_ols(St9, ie, ["ESGID_L1", "TRANS_L1"], CONTROLS),
               focal=["ESGID_L1", "TRANS_L1"],
               label=f"Table 9 (lag) {mlbl}: {ie} ~ ESGID_L1 + TRANS_L1")
else:
    print("[Skipped Table 9] Set AGENCY_COLS to the 6 agency columns in ESG分歧34.xls.")
    print("  Available columns:", list(ESG341.columns))


# =====================================================================================
# ROBUSTNESS: replace ESGR with ESGID (Table 9) -- SAME lagged mediation spec
# =====================================================================================


########## Table 9: ESGID (iterative imputation + PCA, lagged mediation) ##########
firm-years kept (>=1 rating) = 19103 | missing rating cells imputed = 78552
     Prop.%   Cum.%
PC1   76.62   76.62
PC2   11.67   88.29
PC3    4.77   93.06
PC4    4.31   97.38
PC5    2.11   99.49
PC6    0.51  100.00
Components retained (cum >= 70%): k = 1 | weights = [1.]
corr(ESGID, ESGR) = 0.654
ESGID lagged sample N = 15064

=== Table 9 (lag) Model 1: TRANS_L1 ~ ESGID_L1 | N=15064 | adj_R2=0.5878 ===
            coef      t    p  sig
var                              
ESGID_L1  7.6537  4.603  0.0  ***

=== Table 9 (lag) Model 2: IE1 ~ ESGID_L1 + TRANS_L1 | N=15064 | adj_R2=0.1544 ===
            coef       t       p  sig
var                                  
ESGID_L1  0.2265  4.0258  0.0001  ***
TRANS_L1  0.0011  2.6222  0.0087  ***

=== Table 9 (lag) Model 3: IE2 ~ ESGID_L1 + TRANS_L1 | N=15064 | adj_R2=0.0845 ===
            coef       t       p  sig
var                                  
ESGID_L1  

### [16] Table 9 -- ESGID (lagged)

In [20]:
# %% [16] Table 9 -- ESGID with iterative (MICE/EM-style) imputation, lagged ----------
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from statsmodels.multivariate.pca import PCA as smPCA

print("\n########## Table 9: ESGID (iterative imputation + PCA, lagged mediation) ##########")
AGENCY_COLS  = ["华证", "Wind", "富时罗素", "盟浪", "商道融绿", "msci"]  
VAR_THRESHOLD = 0.70  

if all(c in ESG341.columns for c in AGENCY_COLS):
    _ag = ESG341[["Symbol", "Year"] + AGENCY_COLS].drop_duplicates(["Symbol", "Year"])
    St9 = Statement.merge(_ag, on=["Symbol", "Year"], how="left")
    St9 = St9[St9[AGENCY_COLS].notna().any(axis=1)].copy()
    print(f"firm-years kept (>=1 rating) = {len(St9)} | "
          f"missing rating cells imputed = {int(St9[AGENCY_COLS].isna().sum().sum())}")

    imp = IterativeImputer(max_iter=50, random_state=0)
    filled = pd.DataFrame(imp.fit_transform(St9[AGENCY_COLS]),
                          columns=AGENCY_COLS, index=St9.index)

    pc = smPCA(filled, ncomp=len(AGENCY_COLS), standardize=True, method="eig")
    prop = np.asarray(pc.eigenvals).ravel(); prop = prop / prop.sum()
    cum = np.cumsum(prop)

    k = int(np.argmax(cum >= VAR_THRESHOLD)) + 1 if (cum >= VAR_THRESHOLD).any() else len(prop)
    w = prop[:k] / prop[:k].sum()                  
    esgid = pc.factors.iloc[:, :k].values @ w      

    if np.corrcoef(esgid, St9["ESGR"])[0, 1] < 0:
        esgid = -esgid
    St9["ESGID"] = winsorize(pd.Series(esgid, index=St9.index))

    print(pd.DataFrame({"Prop.%": prop * 100, "Cum.%": cum * 100},
                       index=[f"PC{i+1}" for i in range(len(AGENCY_COLS))]).round(2))
    print(f"Components retained (cum >= {VAR_THRESHOLD:.0%}): k = {k} | weights = {np.round(w, 3)}")
    print(f"corr(ESGID, ESGR) = {np.corrcoef(St9['ESGID'], St9['ESGR'])[0, 1]:.3f}")

    St9 = add_firm_lags(St9, ["ESGID"], k=1)
    print(f"ESGID lagged sample N = {St9.dropna(subset=['ESGID_L1', 'TRANS_L1']).shape[0]}")

    report(fe_ols(St9, "TRANS_L1", ["ESGID_L1"], CONTROLS),
           focal=["ESGID_L1"], label="Table 9 (lag) Model 1: TRANS_L1 ~ ESGID_L1")
    for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
        report(fe_ols(St9, ie, ["ESGID_L1", "TRANS_L1"], CONTROLS),
               focal=["ESGID_L1", "TRANS_L1"],
               label=f"Table 9 (lag) {mlbl}: {ie} ~ ESGID_L1 + TRANS_L1")
else:
    print("[Skipped Table 9] Set AGENCY_COLS to the 6 agency columns in ESG分歧34.xls.")
    print("  Available columns:", list(ESG341.columns))


# =====================================================================================
# ENDOGENEITY: 2SLS / diagnostics / system GMM (Table 10) -- contemporaneous, no TRANS
# =====================================================================================


########## Table 9: ESGID (iterative imputation + PCA, lagged mediation) ##########
firm-years kept (>=1 rating) = 19103 | missing rating cells imputed = 78552
     Prop.%   Cum.%
PC1   76.62   76.62
PC2   11.67   88.29
PC3    4.77   93.06
PC4    4.31   97.38
PC5    2.11   99.49
PC6    0.51  100.00
Components retained (cum >= 70%): k = 1 | weights = [1.]
corr(ESGID, ESGR) = 0.654
ESGID lagged sample N = 15064

=== Table 9 (lag) Model 1: TRANS_L1 ~ ESGID_L1 | N=15064 | adj_R2=0.5878 ===
            coef      t    p  sig
var                              
ESGID_L1  7.6537  4.603  0.0  ***

=== Table 9 (lag) Model 2: IE1 ~ ESGID_L1 + TRANS_L1 | N=15064 | adj_R2=0.1544 ===
            coef       t       p  sig
var                                  
ESGID_L1  0.2265  4.0258  0.0001  ***
TRANS_L1  0.0011  2.6222  0.0087  ***

=== Table 9 (lag) Model 3: IE2 ~ ESGID_L1 + TRANS_L1 | N=15064 | adj_R2=0.0845 ===
            coef       t       p  sig
var                                  
ESGID_L1  

### [17] Table 10 -- 2SLS / diagnostics / GMM

In [21]:
# %% [17] Table 10 -- 2SLS / diagnostics / GMM ----------------------------------------
IVS = ["mean1", "mean2", "mean3"]   # industry-year / province-year / industry-province-year mean ESG


def run_2sls(df, y, endog="ESGR", ivs=IVS, controls=CONTROLS):
    sub = df.dropna(subset=[y, endog] + ivs + controls).copy()
    exog = sm.add_constant(sub[controls])
    res = IV2SLS(sub[y], exog, sub[[endog]], sub[ivs]).fit(cov_type="robust")
    return res, sub


print("\n########## Table 10 Panel B: 2SLS (focal coef only) ##########")
res_2sls = {}
for ie in ["IE1", "IE2", "IE3"]:
    r, _ = run_2sls(Statement, ie)
    res_2sls[ie] = r
    print(f"  {ie}: ESG_2SLS = {r.params['ESGR']:.4f} "
          f"(t={r.tstats['ESGR']:.2f}, p={r.pvalues['ESGR']:.4f})")

print("\n########## Table 10 Panel A: diagnostics (IE1) ##########")
r1 = res_2sls["IE1"]
try:
    print(f"Durbin score:       {r1.durbin().stat:.3f} (p={r1.durbin().pval:.4f})")
    wh = r1.wu_hausman()
    print(f"Wu-Hausman:         {wh.stat:.3f} (p={wh.pval:.4f})")
except Exception as e:
    print("endogeneity:", e)
try:
    print(f"Sargan:             {r1.sargan.stat:.3f} (p={r1.sargan.pval:.4f})")
    print(f"Basmann:            {r1.basmann.stat:.3f} (p={r1.basmann.pval:.4f})")
except Exception as e:
    print("overid:", e)
print("First-stage weak-IV diagnostics:")
print(r1.first_stage.diagnostics.round(4))

print("\n########## Table 10 Panel C: two-step system GMM ##########")
print("(pydynpd prints the full table; only the L.IE and ESGR rows are of interest.)")
from pydynpd import regression as dynpd


def run_sys_gmm(df, y, esg="ESGR", controls=CONTROLS):
    g = df[["Symbol", "Year", y, esg] + controls].dropna().copy()
    g["id"] = g["Symbol"].astype("category").cat.codes
    g = g.sort_values(["id", "Year"])
    ctrl = " ".join(controls)
    cmd = (f"{y} L1.{y} {esg} {ctrl} | "
           f"gmm({y}, 2:4) gmm({esg}, 2:4) iv({ctrl}) | timedumm collapse")
    return dynpd.abond(cmd, g, ["id", "Year"])


for ie in ["IE1", "IE2", "IE3"]:
    print(f"\n----- {ie} -----")
    try:
        run_sys_gmm(Statement, ie)
    except Exception as e:
        print(f"[{ie}] GMM failed: {e} (need numpy<2 and a long enough panel)")


# =====================================================================================
# HETEROGENEITY (Tables 11-15) -- ALL use the lagged mediation spec
#   focal: ESGR_L1 + TRANS_L1     (lags already built on the full panel in [10])
# =====================================================================================


########## Table 10 Panel B: 2SLS (focal coef only) ##########
  IE1: ESG_2SLS = 0.0074 (t=9.59, p=0.0000)
  IE2: ESG_2SLS = 0.0061 (t=9.84, p=0.0000)
  IE3: ESG_2SLS = 0.0060 (t=9.72, p=0.0000)

########## Table 10 Panel A: diagnostics (IE1) ##########
Durbin score:       230.523 (p=0.0000)
Wu-Hausman:         233.106 (p=0.0000)
Sargan:             240.133 (p=0.0000)
Basmann:            242.936 (p=0.0000)
First-stage weak-IV diagnostics:
      rsquared  partial.rsquared  shea.rsquared     f.stat  f.pval   f.dist
ESGR    0.3243            0.2072         0.2072  4886.1839     0.0  chi2(3)

########## Table 10 Panel C: two-step system GMM ##########
(pydynpd prints the full table; only the L.IE and ESGR rows are of interest.)

----- IE1 -----
 Dynamic panel-data estimation, two-step system GMM
 Group variable: id                               Number of obs = 11943   
 Time variable: Year                              Min obs per group: 0    
 Number of instruments = 34                   

### [18.1] Table 11 -- corporate life cycle (lagged)

In [22]:
# %% [18.1] Table 11 -- corporate life cycle (lagged) --------------------------------
print("\n########## Table 11: corporate life cycle (lagged) ##########")
try:
    CLC = pd.read_excel("CLC.xlsx")
    CLC["Symbol"] = pad_symbol(CLC["Symbol"])
    CLC["Year"] = pd.to_numeric(CLC["Year"], errors="coerce").astype("Int64").astype(int)
    CLC = yrfilter(CLC)[["Symbol", "Year", "企业生命周期"]]
    CLC["stage"] = CLC["企业生命周期"].map({"成长期": 3, "成熟期": 2, "衰退期": 1})

    St_clc = lj(Statement, CLC[["Symbol", "Year", "stage"]]).dropna(subset=["stage"])
    stage_name = {3: "growth", 2: "mature", 1: "decline"}
    for ie in ["IE1", "IE2", "IE3"]:
        for s in (3, 2, 1):
            sub = St_clc[St_clc["stage"] == s]
            report(fe_ols(sub, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
                   focal=["ESGR_L1", "TRANS_L1"],
                   label=f"{ie} | {stage_name[s]} (N={len(sub)})")
except FileNotFoundError:
    print("[Skipped Table 11] CLC.xlsx not found.")


########## Table 11: corporate life cycle (lagged) ##########

=== IE1 | growth (N=2604) | N=1706 | adj_R2=0.1122 ===
            coef       t       p sig
var                                 
ESGR_L1   0.0036  2.1719  0.0299  **
TRANS_L1  0.0031  2.3176  0.0205  **

=== IE1 | mature (N=6388) | N=4907 | adj_R2=0.1089 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0026  3.7895  0.0002  ***
TRANS_L1  0.0005  0.5775  0.5636     

=== IE1 | decline (N=9808) | N=8356 | adj_R2=0.1672 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0019  4.3283  0.0000  ***
TRANS_L1  0.0006  1.4132  0.1576     

=== IE2 | growth (N=2604) | N=1706 | adj_R2=0.0960 ===
            coef       t       p sig
var                                 
ESGR_L1   0.0015  0.9008  0.3677    
TRANS_L1  0.0002  0.1748  0.8612    

=== IE2 | mature (N=6388) | N=4907 | adj_R2=0.0539 ===
            coef       t       p sig
var                

### [18.2] Table 12 -- over/under investment (H3, lagged)

In [23]:
# %% [18.2] Table 12 -- over/under investment (H3, lagged) ---------------------------
# Over_or_Under_Investment: 1 = overinvestment, 0 = underinvestment (Richardson 2006).
print("\n########## Table 12: over / under investment (H3, lagged) ##########")
over = Statement[Statement["Over_or_Under_Investment"] == 1]
under = Statement[Statement["Over_or_Under_Investment"] == 0]
for ie in ["IE1", "IE2", "IE3"]:
    report(fe_ols(over, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
           focal=["ESGR_L1", "TRANS_L1"], label=f"Overinvestment | {ie} (N={len(over)})")
    report(fe_ols(under, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
           focal=["ESGR_L1", "TRANS_L1"], label=f"Underinvestment | {ie} (N={len(under)})")


########## Table 12: over / under investment (H3, lagged) ##########

=== Overinvestment | IE1 (N=8077) | N=6307 | adj_R2=0.1371 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0030  4.2730  0.0000  ***
TRANS_L1  0.0027  3.3759  0.0007  ***

=== Underinvestment | IE1 (N=11026) | N=8757 | adj_R2=0.2257 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0022  5.3756  0.0000  ***
TRANS_L1 -0.0002 -0.5249  0.5997     

=== Overinvestment | IE2 (N=8077) | N=6307 | adj_R2=0.1358 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0019  2.8009  0.0051  ***
TRANS_L1 -0.0006 -0.8117  0.4170     

=== Underinvestment | IE2 (N=11026) | N=8757 | adj_R2=0.1556 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0014  5.3011  0.0000  ***
TRANS_L1  0.0007  2.8238  0.0047  ***

=== Overinvestment | IE3 (N=8077) | N=6307 | adj_R2=0.1333 ==

### [18.3] Table 13 -- property rights SOE / non-SOE (lagged)

In [24]:
# %% [18.3] Table 13 -- property rights SOE / non-SOE (lagged) -----------------------
print("\n########## Table 13: property-rights heterogeneity (lagged) ##########")
soe = Statement[Statement["SOE"] == 1]
nonsoe = Statement[Statement["SOE"] == 0]
for ie in ["IE1", "IE2", "IE3"]:
    report(fe_ols(soe, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
           focal=["ESGR_L1", "TRANS_L1"], label=f"SOEs | {ie} (N={len(soe)})")
    report(fe_ols(nonsoe, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
           focal=["ESGR_L1", "TRANS_L1"], label=f"non-SOEs | {ie} (N={len(nonsoe)})")


########## Table 13: property-rights heterogeneity (lagged) ##########

=== SOEs | IE1 (N=1609) | N=1207 | adj_R2=0.2337 ===
            coef       t       p sig
var                                 
ESGR_L1  -0.0001 -0.0533  0.9575    
TRANS_L1  0.0017  1.0637  0.2875    

=== non-SOEs | IE1 (N=17494) | N=13857 | adj_R2=0.1565 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0027  6.3242  0.0000  ***
TRANS_L1  0.0010  2.3987  0.0165   **

=== SOEs | IE2 (N=1609) | N=1207 | adj_R2=0.0964 ===
            coef       t       p sig
var                                 
ESGR_L1   0.0011  0.9743  0.3299    
TRANS_L1  0.0014  0.8251  0.4093    

=== non-SOEs | IE2 (N=17494) | N=13857 | adj_R2=0.0871 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0017  4.4771  0.0000  ***
TRANS_L1  0.0003  0.8361  0.4031     

=== SOEs | IE3 (N=1609) | N=1207 | adj_R2=0.0844 ===
            coef       t       p sig
var      

### [18.4] Table 14 -- industry heterogeneity (lagged)

In [25]:
# %% [18.4] Table 14 -- industry heterogeneity (lagged) ------------------------------
HIGH_POLLUTION = {"B06","B07","B08","B09","C17","C19","C22","C25","C26","C28",
                  "C29","C30","C31","C32","D44"}
HIGH_TECH = {"C25","C26","C27","C28","C29","C31","C32","C34","C35","C36","C37",
             "C38","C39","C40","C41","I63","I64","I65","M73"}
LABOR_INT = {"A01","A02","A03","A05","B06","B08","B09","C13","C14","C15","C17",
             "C18","C19","C20","C21","C23","C24","C32","C34","D46","E48","E49",
             "E50","F51","F52","G53","G54","G58","G59","I63","I64","K70","L72",
             "M73","M75","N78","P82","R85","R87","S90"}
TECH_INT = {"N77","C36","M74","I65","C33","C35","C27","C29","C39","C38","C37",
            "C41","C40"}
CAP_INT = {"G56","D44","A04","B11","D45","B07","C22","C31","G55","C30","R86",
           "C28","C26","C25"}


def industry_letter(x):
    s = str(x).strip().upper()
    return s[:3] if len(s) >= 3 else s


print("\n########## Table 14: industry heterogeneity (lagged) ##########")
if "Industry" in Statement.columns:
    St = Statement.copy()
    St["icode"] = St["Industry"].map(industry_letter)
    groups = {
        "High pollution":       St[St["icode"].isin(HIGH_POLLUTION)],
        "Low pollution":        St[~St["icode"].isin(HIGH_POLLUTION)],
        "High-tech":            St[St["icode"].isin(HIGH_TECH)],
        "Traditional":          St[~St["icode"].isin(HIGH_TECH)],
        "Labor-intensive":      St[St["icode"].isin(LABOR_INT)],
        "Technology-intensive": St[St["icode"].isin(TECH_INT)],
        "Capital-intensive":    St[St["icode"].isin(CAP_INT)],
    }
    for name, sub in groups.items():
        if len(sub) > 50:
            for ie in ["IE1", "IE2", "IE3"]:
                report(fe_ols(sub, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
                       focal=["ESGR_L1", "TRANS_L1"], label=f"{name} | {ie} (N={len(sub)})")
else:
    print("[Skipped Table 14] missing the 'Industry' code column.")


########## Table 14: industry heterogeneity (lagged) ##########

=== High pollution | IE1 (N=3382) | N=2647 | adj_R2=0.0982 ===
            coef       t       p sig
var                                 
ESGR_L1   0.0018  1.8942  0.0582   *
TRANS_L1  0.0005  0.4754  0.6345    

=== High pollution | IE2 (N=3382) | N=2647 | adj_R2=0.0616 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0024  2.8826  0.0039  ***
TRANS_L1  0.0001  0.1153  0.9082     

=== High pollution | IE3 (N=3382) | N=2647 | adj_R2=0.0599 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0021  2.5948  0.0095  ***
TRANS_L1  0.0003  0.2694  0.7876     

=== Low pollution | IE1 (N=15721) | N=12417 | adj_R2=0.1712 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0025  5.6198  0.0000  ***
TRANS_L1  0.0013  3.0042  0.0027  ***

=== Low pollution | IE2 (N=15721) | N=12417 | adj_R2=0.0851 ===
         

### [18.5] Table 15 -- Confucianism (lagged)

In [26]:
# %% [18.5] Table 15 -- Confucianism (lagged) ----------------------------------------
print("\n########## Table 15: Confucianism (lagged) ##########")
try:
    CONFU = pd.read_excel("儒家文化.xlsx")
    CONFU["Symbol"] = pad_symbol(CONFU["Symbol"])
    CONFU["Year"] = pd.to_numeric(CONFU["Year"], errors="coerce").astype("Int64").astype(int)
    CONFU = yrfilter(CONFU)
    confu_cols = [f"confu{i}" for i in range(1, 8)]
    CONFU["confu_mean"] = CONFU[confu_cols].mean(axis=1)

    St_cf = lj(Statement, CONFU[["Symbol", "Year", "confu_mean"]]).dropna(subset=["confu_mean"])
    med = St_cf["confu_mean"].median()
    St_cf["confu_high"] = (St_cf["confu_mean"] >= med).astype(int)

    for grp, lbl in [(1, "High Confucian"), (0, "Low Confucian")]:
        sub = St_cf[St_cf["confu_high"] == grp]
        for ie in ["IE1", "IE2", "IE3"]:
            report(fe_ols(sub, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS),
                   focal=["ESGR_L1", "TRANS_L1"], label=f"{lbl} | {ie} (N={len(sub)})")
except FileNotFoundError:
    print("[Skipped Table 15] 儒家文化.xlsx not found.")


print("\n===== All analyses complete =====")


########## Table 15: Confucianism (lagged) ##########

=== High Confucian | IE1 (N=9549) | N=7518 | adj_R2=0.1477 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0029  4.7803  0.0000  ***
TRANS_L1  0.0018  3.3431  0.0008  ***

=== High Confucian | IE2 (N=9549) | N=7518 | adj_R2=0.0815 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0018  3.4477  0.0006  ***
TRANS_L1  0.0006  1.1943  0.2324     

=== High Confucian | IE3 (N=9549) | N=7518 | adj_R2=0.0802 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0015  2.9056  0.0037  ***
TRANS_L1  0.0005  0.9931  0.3207     

=== Low Confucian | IE1 (N=9549) | N=7544 | adj_R2=0.1671 ===
            coef       t       p  sig
var                                  
ESGR_L1   0.0022  4.1366  0.0000  ***
TRANS_L1  0.0004  0.7026  0.4823     

=== Low Confucian | IE2 (N=9549) | N=7544 | adj_R2=0.0907 ===
            coef   